# Train And Save SAE From A Trained Transformer

This notebook only does path setup, SAE training, and saving the SAE artifact for later analysis.

In [9]:
import os
import subprocess
from pathlib import Path

import torch

print(f'Current notebook cwd: {Path.cwd()}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device count: {torch.cuda.device_count()}')
    print(f'Current CUDA device: {torch.cuda.get_device_name(0)}')
else:
    print('Warning: CUDA not available, will run on CPU unless you set device manually.')

!nvidia-smi

Current notebook cwd: /home/ponsin
CUDA available: True
CUDA device count: 1
Current CUDA device: NVIDIA H100
Mon Mar 23 11:03:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100                    On  |   00000000:04:00.0 Off |                    0 |
| N/A   28C    P0             64W /  700W |       5MiB /  95830MiB |      0%      Default |
|                             

## Configure Paths, SAE Hyperparameters, And SAE Data Splits

Use the consolidated transformer output file produced by `main.py` (typically ending with `.pt`, and in your Slurm naming often `.pkl.pt`).
Then choose where to save the trained SAE artifact, plus the train/eval RHM sizes used for SAE training vs later analysis.

In [15]:
# Required input artifact from transformer training (single consolidated file from main.py)
train_output = Path('/work/pcsl/ponsin/Mean_Transformer/Transformer_for_SAE/v_16_L_3_m_4/RESULT_TRFCLASS_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1.pkl.pt')

# Path to the exact train_sae.py script you want to run
train_sae_script = Path('/home/ponsin/SAE-on-RHM/train_sae.py')

# SAE training settings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sae_activation_source = 'one_token'  # 'all_tokens', 'cls_token', or 'one_token'
sae_token_idx = 0                     # only used when sae_activation_source = 'one_token'
sae_log_points = 2**9

# RHM sizes/seeds for SAE pipeline
sae_train_size = 2**14          # data used to train SAE
sae_eval_size = 2**14           # separate data reserved for later SAE analysis
sae_train_seed_sample = None    # None => transformer seed + 1 (different data, same rules)
sae_eval_seed_sample = None     # None => SAE train seed + 1

# Per-layer SAE hyperparameters (edit independently per layer)
layer_sae_hparams = {
    0: {
        'sae_latent_dim': 20 * 512,
        'sae_lambda_l1': 0.1,
        'sae_lr': 5e-5,
        'sae_steps': 2**13,
        'sae_sample_batch_size': 2**7,
        'sae_batch_limit': 0,
    },
    1: {
        'sae_latent_dim': 20 * 512,
        'sae_lambda_l1': 0.1,
        'sae_lr': 5e-5,
        'sae_steps': 2**13,
        'sae_sample_batch_size': 2**7,
        'sae_batch_limit': 0,
    },
    2: {
        'sae_latent_dim': 20 * 512,
        'sae_lambda_l1': 0.1,
        'sae_lr': 5e-5,
        'sae_steps': 2**13,
        'sae_sample_batch_size': 2**7,
        'sae_batch_limit': 0,
    },
}

def _activation_tag(source, tok_idx):
    """Short label used in filenames and print output."""
    if source == 'all_tokens':
        return 'alltok'
    if source == 'cls_token':
        return 'cls'
    if source == 'one_token':
        return f'tok{tok_idx}'
    return source

_atag = _activation_tag(sae_activation_source, sae_token_idx)

# Required output artifacts, one per layer
sae_output_paths = {}
for layer, hp in layer_sae_hparams.items():
    sae_output_paths[layer] = Path(
        f"/work/pcsl/ponsin/Mean_Transformer/SAE/v_16_L_3_m_4/"
        f"SAE_layer_{layer}_{_atag}_v_16_L_3_m=4_P_{sae_train_size}"
        f"_l1_{hp['sae_lambda_l1']}_ldim_{hp['sae_latent_dim']}"
        f"_lr_{hp['sae_lr']}_steps_{hp['sae_steps']}_bsize_{hp['sae_sample_batch_size']}.pt"
    )

print(f'train_output: {train_output}')
print(f'train_sae_script: {train_sae_script}')
print(f'sae_activation_source: {sae_activation_source}')
if sae_activation_source == 'one_token':
    print(f'sae_token_idx: {sae_token_idx}')
print(f'sae_train_size: {sae_train_size}')
print(f'sae_eval_size: {sae_eval_size}')
print(f'sae_train_seed_sample: {sae_train_seed_sample}')
print(f'sae_eval_seed_sample: {sae_eval_seed_sample}')
for layer in sorted(sae_output_paths):
    print(f'layer {layer} output path: {sae_output_paths[layer]}')

train_output: /work/pcsl/ponsin/Mean_Transformer/Transformer_for_SAE/v_16_L_3_m_4/RESULT_TRFCLASS_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1.pkl.pt
train_sae_script: /home/ponsin/SAE-on-RHM/train_sae.py
sae_activation_source: one_token
sae_token_idx: 0
sae_train_size: 16384
sae_eval_size: 16384
sae_train_seed_sample: None
sae_eval_seed_sample: None
layer 0 output path: /work/pcsl/ponsin/Mean_Transformer/SAE/v_16_L_3_m_4/SAE_layer_0_tok0_v_16_L_3_m=4_P_16384_l1_0.1_ldim_10240_lr_5e-05_steps_8192_bsize_128.pt
layer 1 output path: /work/pcsl/ponsin/Mean_Transformer/SAE/v_16_L_3_m_4/SAE_layer_1_tok0_v_16_L_3_m=4_P_16384_l1_0.1_ldim_10240_lr_5e-05_steps_8192_bsize_128.pt
layer 2 output path: /work/pcsl/ponsin/Mean_Transformer/SAE/v_16_L_3_m_4/SAE_layer_2_tok0_v_16_L_3_m=4_P_16384_l1_0.1_ldim_10240_lr_5e-05_steps_8192_bsize_128.pt


In [16]:
def _assert_exists(path_obj, name):
    if path_obj is None:
        raise ValueError(f'{name} is None')
    if not Path(path_obj).exists():
        raise FileNotFoundError(f'{name} not found: {path_obj}')

_assert_exists(train_output, 'train_output')
_assert_exists(train_sae_script, 'train_sae_script')

# Ensure this artifact carries the exact RHM rules used in transformer training.
blob = torch.load(train_output, map_location='cpu')
if not isinstance(blob, dict) or 'output' not in blob:
    raise ValueError(
        'train_output must be a consolidated main.py artifact containing an output dict.'
    )
if 'rules' not in blob['output']:
    raise ValueError(
        'train_output is missing output.rules. Re-run transformer training with the updated transformer script.'
    )
print('Verified train_output contains output.rules for fixed-RHM SAE training.')


def build_sae_cmd(layer_id, sae_output_path, hp):
    cmd = [
        'python', str(train_sae_script),
        '--train_output', str(train_output),
        '--outname', str(sae_output_path),
        '--device', device,
        '--sae_layer', str(int(layer_id)),
        '--sae_activation_source', str(sae_activation_source),
        '--sae_lambda_l1', str(float(hp['sae_lambda_l1'])),
        '--sae_lr', str(float(hp['sae_lr'])),
        '--sae_steps', str(int(hp['sae_steps'])),
        '--sae_sample_batch_size', str(int(hp['sae_sample_batch_size'])),
        '--sae_batch_limit', str(int(hp['sae_batch_limit'])),
        '--sae_log_points', str(int(sae_log_points)),
        '--sae_train_size', str(int(sae_train_size)),
        '--sae_eval_size', str(int(sae_eval_size)),
    ]

    if sae_activation_source == 'one_token':
        cmd += ['--sae_token_idx', str(int(sae_token_idx))]
    if hp.get('sae_latent_dim') is not None:
        cmd += ['--sae_latent_dim', str(int(hp['sae_latent_dim']))]
    if sae_train_seed_sample is not None:
        cmd += ['--sae_train_seed_sample', str(int(sae_train_seed_sample))]
    if sae_eval_seed_sample is not None:
        cmd += ['--sae_eval_seed_sample', str(int(sae_eval_seed_sample))]

    return cmd


sae_cmds = {
    layer: build_sae_cmd(layer, sae_output_paths[layer], layer_sae_hparams[layer])
    for layer in sorted(layer_sae_hparams)
}

print(f'Commands ({_atag}, per-layer):')
for layer in sorted(sae_cmds):
    print(f'layer {layer}:')
    print(' '.join(sae_cmds[layer]))

Verified train_output contains output.rules for fixed-RHM SAE training.
Commands (tok0, per-layer):
layer 0:
python /home/ponsin/SAE-on-RHM/train_sae.py --train_output /work/pcsl/ponsin/Mean_Transformer/Transformer_for_SAE/v_16_L_3_m_4/RESULT_TRFCLASS_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1.pkl.pt --outname /work/pcsl/ponsin/Mean_Transformer/SAE/v_16_L_3_m_4/SAE_layer_0_tok0_v_16_L_3_m=4_P_16384_l1_0.1_ldim_10240_lr_5e-05_steps_8192_bsize_128.pt --device cuda --sae_layer 0 --sae_activation_source one_token --sae_lambda_l1 0.1 --sae_lr 5e-05 --sae_steps 8192 --sae_sample_batch_size 128 --sae_batch_limit 0 --sae_print_freq 512 --sae_train_size 16384 --sae_eval_size 16384 --sae_token_idx 0 --sae_latent_dim 10240
layer 1:
python /home/ponsin/SAE-on-RHM/train_sae.py --train_output /work/pcsl/ponsin/Mean_Transformer/Transformer_for_SAE/v_16_L_3_m_4/RESULT_TRFCLASS_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1.pkl.pt --outname /work/pcsl/ponsin/Mean_Transformer/SAE/v_1

/tmp/2683569/ipykernel_1229286/1600372130.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  blob = torch.load(train_output, map_location='cpu')


In [17]:
def run_and_check(cmd, label):
    print(f'Running SAE training: {label}')
    result = subprocess.run(
        cmd,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'{label} failed with return code {result.returncode}')
    print(f'{label} finished successfully.')
    return result.stdout

sae_train_logs = {}
for layer in sorted(sae_cmds):
    sae_train_logs[layer] = run_and_check(sae_cmds[layer], f'{_atag} SAE layer {layer}')

print('All per-layer SAE trainings finished successfully.')

Running SAE training: tok0 SAE layer 0
SAE data split: train_size=16384 (seed_sample=16129767), eval_size=16384 (seed_sample=16129768)
Transformer training seed_sample=16129766
Using fixed RHM rules loaded from training artifact.
# parameters: 9464848
sae layer 0 step 0/8192 total=173.423248 recon=123.501343 sparse=499.219055 active_fraction=0.502850 dead_features=475
sae layer 0 step 512/8192 total=35.386112 recon=9.635447 sparse=257.506653
sae layer 0 step 1024/8192 total=33.033424 recon=7.768342 sparse=252.650833
sae layer 0 step 1536/8192 total=33.347469 recon=7.370784 sparse=259.766846
sae layer 0 step 2048/8192 total=33.099510 recon=6.979331 sparse=261.201782
sae layer 0 step 2560/8192 total=32.095451 recon=6.076657 sparse=260.187927
sae layer 0 step 3072/8192 total=31.687244 recon=5.825367 sparse=258.618774
sae layer 0 step 3584/8192 total=31.462017 recon=5.839067 sparse=256.229492
sae layer 0 step 4096/8192 total=30.573814 recon=5.426690 sparse=251.471252
sae layer 0 step 4608/

In [18]:
for layer in sorted(sae_output_paths):
    p = Path(sae_output_paths[layer])
    print(f'{_atag} SAE layer {layer} output path: {p.resolve()}')
    if not p.exists():
        raise FileNotFoundError(f'Could not find {_atag} SAE output for layer {layer} at {p}')

print('Per-layer SAE artifacts are ready for loading in a later step.')

tok0 SAE layer 0 output path: /work/pcsl/ponsin/Mean_Transformer/SAE/v_16_L_3_m_4/SAE_layer_0_tok0_v_16_L_3_m=4_P_16384_l1_0.1_ldim_10240_lr_5e-05_steps_8192_bsize_128.pt
tok0 SAE layer 1 output path: /work/pcsl/ponsin/Mean_Transformer/SAE/v_16_L_3_m_4/SAE_layer_1_tok0_v_16_L_3_m=4_P_16384_l1_0.1_ldim_10240_lr_5e-05_steps_8192_bsize_128.pt
tok0 SAE layer 2 output path: /work/pcsl/ponsin/Mean_Transformer/SAE/v_16_L_3_m_4/SAE_layer_2_tok0_v_16_L_3_m=4_P_16384_l1_0.1_ldim_10240_lr_5e-05_steps_8192_bsize_128.pt
Per-layer SAE artifacts are ready for loading in a later step.


In [19]:
# Compact SAE analysis on a fresh evaluation split
import copy
import sys
from pathlib import Path

import torch

repo_root = Path(train_sae_script).resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import init
import models
from datasets.random_hierarchy_model import sample_trees

# -----------------------------
# Analysis configuration
# -----------------------------
analysis_eval_size = 2**15
analysis_eval_seed_sample = 98765432
analysis_batch_size = 256
analysis_device = device

In [20]:
def load_transformer_and_loader(train_output_path, eval_size, eval_seed, batch_size, device_name):
    blob = torch.load(train_output_path, map_location='cpu')
    if not isinstance(blob, dict) or 'config' not in blob or 'output' not in blob:
        raise ValueError('train_output must contain config and output dictionaries.')
    if 'model' not in blob['output'] or 'rules' not in blob['output'] or blob['output']['rules'] is None:
        raise ValueError('train_output must contain output.model and output.rules.')

    cfg = copy.deepcopy(blob['config'])
    rules = blob['output']['rules']

    trees_eval = sample_trees(num_data=int(eval_size), rules=rules, prior=None, probs=None, seed=int(eval_seed))

    data_cfg = copy.deepcopy(cfg)
    data_cfg.train_size = int(eval_size)
    data_cfg.test_size = 0
    data_cfg.batch_size = max(1, min(int(batch_size), int(eval_size)))
    loader, _ = init.init_data(trees_eval[cfg.num_layers], trees_eval[0], data_cfg)

    model = init.init_model(cfg)
    model.load_state_dict(blob['output']['model'])
    model = model.to(device_name).eval()
    for p in model.parameters():
        p.requires_grad = False

    return model, loader


def load_sae_set(ckpt_paths_by_layer, input_dim, device_name):
    modules = {}
    sae_layers = []

    for layer in sorted(ckpt_paths_by_layer):
        ckpt_path = ckpt_paths_by_layer[layer]
        ckpt = torch.load(ckpt_path, map_location='cpu')
        if 'sae_state' not in ckpt or 'sae_layers' not in ckpt:
            raise ValueError(f'Invalid SAE checkpoint format: {ckpt_path}')

        ckpt_layers = [int(x) for x in ckpt['sae_layers']]
        if len(ckpt_layers) != 1 or ckpt_layers[0] != int(layer):
            raise ValueError(
                f'Expected checkpoint for layer {layer}, got layers={ckpt_layers} in {ckpt_path}'
            )

        layer_key = ckpt_layers[0]
        state = ckpt['sae_state'].get(layer_key) or ckpt['sae_state'].get(str(layer_key))
        metrics = (ckpt.get('sae_metrics', {}).get(layer_key)
                   or ckpt.get('sae_metrics', {}).get(str(layer_key))
                   or {})
        if state is None:
            raise ValueError(f'Missing SAE state for layer {layer_key} in {ckpt_path}')

        latent_dim = int(metrics.get('latent_dim') or state['encoder.weight'].shape[0])
        sae = models.SparseAutoencoder(input_dim=input_dim, latent_dim=latent_dim).to(device_name)
        sae.load_state_dict(state)
        sae.eval()
        for p in sae.parameters():
            p.requires_grad = False

        modules[layer_key] = sae
        sae_layers.append(layer_key)

    return modules, sorted(sae_layers)

In [21]:
def compute_activity_stats(model, loader, sae_modules, sae_layers, mode, device_name, token_idx=0):
    """Compute dead-feature and mean-active-feature statistics over the eval loader.

    A feature is considered active when its value > 0, where the feature value is
    defined as  z_i * ||d_i||  (hidden activation times decoder column norm),
    matching the weighted L1 penalty used in the SAE loss.

    mode: 'all_tokens', 'cls_token', or 'one_token'
    token_idx: 0-based real-token index, only used when mode='one_token'.
               Counts from 0 among the real sequence tokens (skips [CLS] when present).
    """
    assert mode in {'all_tokens', 'cls_token', 'one_token'}
    has_cls_token = hasattr(model, 'cls_token')
    if mode == 'cls_token' and not has_cls_token:
        raise ValueError('cls_token mode is invalid for models without a CLS token.')

    ever_active = {l: torch.zeros(sae_modules[l].latent_dim, dtype=torch.bool, device=device_name) for l in sae_layers}
    active_feature_sum = {l: 0.0 for l in sae_layers}
    token_count = {l: 0 for l in sae_layers}

    # Pre-compute decoder column norms for each SAE layer (shape: latent_dim,)
    dec_norms = {l: sae_modules[l].decoder_feature_norms().to(device_name) for l in sae_layers}

    buffers = {l: [] for l in sae_layers}
    hooks = [model.blocks[l].register_forward_hook(lambda _m, _i, o, layer=l: buffers[layer].append(o.detach()))
             for l in sae_layers]

    with torch.no_grad():
        for x_batch, _ in loader:
            model(x_batch.to(device_name))
            for l in sae_layers:
                if not buffers[l]:
                    continue
                act = buffers[l].pop(0)
                if mode == 'cls_token':
                    act = act[:, :1, :]
                elif mode == 'one_token':
                    offset = 1 if has_cls_token else 0
                    act = act[:, offset + token_idx : offset + token_idx + 1, :]
                elif has_cls_token:
                    act = act[:, 1:, :]
                act = act.reshape(-1, act.size(-1))
                if act.numel() == 0:
                    continue
                _, z = sae_modules[l](act)
                # Feature value: z_i * ||d_i||  (matches the SAE weighted L1 loss)
                features = z * dec_norms[l].unsqueeze(0)   # (N, latent_dim)
                is_active = features > 0
                active_feature_sum[l] += is_active.float().sum().item()
                token_count[l] += is_active.size(0)
                ever_active[l] |= is_active.any(dim=0)

    for h in hooks:
        h.remove()

    return {
        l: {
            'latent_dim': int(sae_modules[l].latent_dim),
            'dead_features': int((~ever_active[l]).sum()),
            'dead_feature_ratio': float((~ever_active[l]).sum()) / max(sae_modules[l].latent_dim, 1),
            'mean_active_features': active_feature_sum[l] / max(token_count[l], 1),
            'mean_active_feature_ratio': active_feature_sum[l] / max(token_count[l] * sae_modules[l].latent_dim, 1),
        }
        for l in sae_layers
    }


def print_stats(title, stats):
    print('=' * 108)
    print(title)
    print('-' * 108)
    print(f"{'layer':>6} | {'latent_dim':>10} | {'dead_features':>13} | {'dead_ratio':>10} | {'mean_active':>12} | {'mean_active_ratio':>17}")
    print('-' * 108)
    for l in sorted(stats):
        s = stats[l]
        print(f"{l:6d} | {s['latent_dim']:10d} | {s['dead_features']:13d} | {s['dead_feature_ratio']:10.6f} | {s['mean_active_features']:12.4f} | {s['mean_active_feature_ratio']:17.6f}")

In [22]:
for p in [train_output, *sae_output_paths.values()]:
    if not Path(p).exists():
        raise FileNotFoundError(f'Missing required artifact: {p}')

model, eval_loader = load_transformer_and_loader(
    train_output_path=train_output,
    eval_size=analysis_eval_size,
    eval_seed=analysis_eval_seed_sample,
    batch_size=analysis_batch_size,
    device_name=analysis_device,
)

all_modules, all_layers = load_sae_set(sae_output_paths, model.embedding_dim, analysis_device)

all_stats = compute_activity_stats(
    model, eval_loader, all_modules, all_layers,
    mode=sae_activation_source, device_name=analysis_device,
    token_idx=sae_token_idx,
)

_tok_info = f' (token {sae_token_idx})' if sae_activation_source == 'one_token' else ''
print(f'Analysis eval size: {analysis_eval_size}, seed: {analysis_eval_seed_sample}, batch_size: {analysis_batch_size}')
print_stats(f'{_atag}{_tok_info} SAE stats (per-layer checkpoints)', all_stats)

/tmp/2683569/ipykernel_1229286/978324559.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  blob = torch.load(train_output_path, map_location='cpu')


# parameters: 9464848


/tmp/2683569/ipykernel_1229286/978324559.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location='cpu')
/home/ponsin/miniconda3/envs/p

Analysis eval size: 32768, seed: 98765432, batch_size: 256
tok0 (token 0) SAE stats (per-layer checkpoints)
------------------------------------------------------------------------------------------------------------
 layer | latent_dim | dead_features | dead_ratio |  mean_active | mean_active_ratio
------------------------------------------------------------------------------------------------------------
     0 |      10240 |          6385 |   0.623535 |     110.0339 |          0.010746
     1 |      10240 |           829 |   0.080957 |     174.0353 |          0.016996
     2 |      10240 |           192 |   0.018750 |     446.9206 |          0.043645


## SAE Impact on Classification Performance

Each SAE is trained in a **separate process per transformer layer**, enabling layer-specific hyperparameters.
In this section, we hook into the transformer and replace each selected block output with its layer-specific SAE reconstruction (encoder -> ReLU -> decoder).

For MeanClassificationTransformer, `all_tokens` corresponds to **all real token positions (0..T-1)**.

**Normalized test error** = `err_with_sae / err_baseline`. A value of 1 means no degradation; larger values indicate reconstruction loss hurts classification.

In [ ]:
import torch.nn.functional as F


def eval_classification(model, loader, device_name, intervention_hooks=None):
    """Run inference and return (accuracy, mean cross-entropy loss).
    intervention_hooks: optional list of (module, hook_fn) applied during forward passes.
    """
    handles = []
    if intervention_hooks:
        for module, fn in intervention_hooks:
            handles.append(module.register_forward_hook(fn))

    correct = total = 0
    total_ce = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device_name), y.to(device_name)
            logits = model(x)
            total_ce += F.cross_entropy(logits, y, reduction='sum').item()
            correct += (logits.argmax(-1) == y).sum().item()
            total += y.size(0)

    for h in handles:
        h.remove()

    return correct / total, total_ce / total


def make_sae_hook(sae, mode, has_cls_token, token_idx=0):
    """Return a forward hook that replaces block output with SAE reconstruction
    for the token positions the SAE was trained on.

    mode: 'all_tokens', 'cls_token', or 'one_token'
    token_idx: 0-based real-token index (only for mode='one_token').
               Counts from 0 among the real sequence tokens, skipping [CLS] when present.
    """
    def hook(_m, _i, output):
        out = output.clone()
        if mode == 'cls_token':
            if not has_cls_token:
                raise ValueError('cls_token mode is invalid for models without a CLS token.')
            flat = out[:, :1, :].reshape(-1, out.size(-1))
            recon, _ = sae(flat)
            out[:, :1, :] = recon.reshape(output[:, :1, :].shape)
        elif mode == 'one_token':
            offset = 1 if has_cls_token else 0
            pos = offset + token_idx
            flat = out[:, pos : pos + 1, :].reshape(-1, out.size(-1))
            recon, _ = sae(flat)
            out[:, pos : pos + 1, :] = recon.reshape(output[:, pos : pos + 1, :].shape)
        else:  # all_tokens
            token_slice = slice(1, None) if has_cls_token else slice(0, None)
            flat = out[:, token_slice, :].reshape(-1, out.size(-1))
            recon, _ = sae(flat)
            out[:, token_slice, :] = recon.reshape(output[:, token_slice, :].shape)
        return out
    return hook


has_cls_token = hasattr(model, 'cls_token')

# Random-chance baseline
random_err = 1.0 - 1.0 / model.num_classes
print(f'Number of classes v={model.num_classes},  random error = {random_err:.6f}')

# Baseline (no SAE intervention)
baseline_acc, baseline_ce = eval_classification(model, eval_loader, analysis_device)
baseline_err = 1.0 - baseline_acc
baseline_std = baseline_err / random_err
print(f'Baseline  acc={baseline_acc:.4f}  err={baseline_err:.6f}  std_err={baseline_std:.4f}  CE={baseline_ce:.4f}\n')

# Per-layer and all-layers-stacked interventions
header = f"{'layer':>6} | {'acc':>8} | {'err':>10} | {'std_err':>9} | {'CE':>8}"
sep = '-' * len(header)

_tok_label = f' (token {sae_token_idx})' if sae_activation_source == 'one_token' else ''
print(f'== {_atag}{_tok_label} SAE ==')
print(header)
print(sep)

for layer in all_layers:
    hook_fn = make_sae_hook(all_modules[layer], sae_activation_source, has_cls_token, token_idx=sae_token_idx)
    acc, ce = eval_classification(
        model, eval_loader, analysis_device,
        intervention_hooks=[(model.blocks[layer], hook_fn)],
    )
    err = 1.0 - acc
    std_err = err / random_err
    print(f"{layer:>6} | {acc:>8.4f} | {err:>10.6f} | {std_err:>9.4f} | {ce:>8.4f}")

# All layers stacked simultaneously
all_hooks = [
    (model.blocks[l], make_sae_hook(all_modules[l], sae_activation_source, has_cls_token, token_idx=sae_token_idx))
    for l in all_layers
]
acc, ce = eval_classification(model, eval_loader, analysis_device, intervention_hooks=all_hooks)
err = 1.0 - acc
std_err = err / random_err
print(sep)
print(f"{'all':>6} | {acc:>8.4f} | {err:>10.6f} | {std_err:>9.4f} | {ce:>8.4f}  <- all layers stacked")
print()

In [12]:
# Latent recovery analysis (phase 1): row-aligned SAE features vs RHM latents
import copy
import torch

from datasets.random_hierarchy_model import sample_trees
from latent_analysis import (
    build_latent_row_table,
    collect_weighted_sae_activations,
    compute_observational_metrics,
)


def load_eval_trees(train_output_path, eval_size, eval_seed):
    blob = torch.load(train_output_path, map_location='cpu')
    cfg = copy.deepcopy(blob['config'])
    rules = blob['output']['rules']
    trees_eval = sample_trees(num_data=int(eval_size), rules=rules, prior=None, probs=None, seed=int(eval_seed))
    return cfg, trees_eval


cfg_eval, trees_eval = load_eval_trees(
    train_output_path=train_output,
    eval_size=analysis_eval_size,
    eval_seed=analysis_eval_seed_sample,
)

activation_pack = collect_weighted_sae_activations(
    model=model,
    sae_by_layer=all_modules,
    inputs=trees_eval[cfg_eval.num_layers].long(),
    model_name=str(cfg_eval.model),
    activation_source='all_tokens',
    batch_size=analysis_batch_size,
    device=torch.device(analysis_device),
)

latent_rows = build_latent_row_table(
    trees=trees_eval,
    index=activation_pack.index,
    tuple_size=int(cfg_eval.tuple_size),
)

# Start with a small target set for a quick, interpretable first pass.
class_values = sorted(torch.unique(trees_eval[0]).tolist())
target_masks = {f'class:v{int(v)}': latent_rows.global_mask(level=0, position=0, value=int(v)) for v in class_values}

leaf_values = sorted(torch.unique(trees_eval[cfg_eval.num_layers]).tolist())
for v in leaf_values[: min(8, len(leaf_values))]:
    target_masks[f'token:v{int(v)}'] = latent_rows.ancestor_mask(level=int(cfg_eval.num_layers), value=int(v))

metrics = compute_observational_metrics(activation_pack.weighted_activations, target_masks)

print(f'Rows={activation_pack.index.num_rows()}  Features={activation_pack.weighted_activations.shape[1]}')
print(f'Targets={len(metrics.target_names)}')

for idx, name in enumerate(metrics.target_names[:10]):
    top_delta = torch.topk(metrics.delta_mean[idx], k=min(5, metrics.delta_mean.shape[1]))
    top_feats = [int(i) for i in top_delta.indices.tolist()]
    top_vals = [float(v) for v in top_delta.values.tolist()]
    print(f'{name:>20} | count={int(metrics.target_counts[idx]):>6} | top_delta_features={top_feats} | delta={top_vals}')

/tmp/2678623/ipykernel_1420978/1536427719.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  blob = torch.load(train_output_path, map_location='cpu')


: 

: 

: 